# **Reinforcement Learning Agent For CNN Hyper Parameter Tunning**

### *Here tune only leraning rate*

- Before implementing Q-learning, first prepare the environment that Q-learning will control
- consider only one hyper parameter for now - **Learning Rate**
- Define RL Components

  **State** (What info agent sees)
  - state = current learning rate
  
  **Action** (What agent can do)
    - increase LR
    - decrease LR
    - keep LR sameincrease LR
    - decrease LR
    - keep LR same

  **Reward** (What agent tries to maximize)
    - reward = validation accuracy
 
  **Environment**
  - Environment = CNN training process

- Agent → chooses LR >>> 
CNN → trains one epoch >>> 
Environment → returns accuracy

**Q(S,A) ← Q(S,A) + α (R + γQ(S′,A′) − Q(S,A))**

S is the current state<br>
A is the action taken by the agent<br>
S' is the next state the agent moves to<br>
A' is the best next action in state S'<br>
R is the reward received for taking action A in state S<br>
γ (Gamma) is the discount factor which balances immediate rewards with future rewards<br>
α (Alpha) is the learning rate determining how much new information affects the old Q-values

In [1]:
import torch # main deep learning library
import torch.nn as nn # contains neural network layers and loss functions
import torch.optim as optim # optimization algorithms (SGD, Adam, etc.)

import torchvision # datasets + transforms for images
import torchvision.transforms as transforms
from torch.utils.data import random_split

import matplotlib

print(torch.__version__)
print(torchvision.__version__)
print(matplotlib.__version__)

2.5.1
0.20.1
3.10.9


In [2]:
# If GPU is available → use it
# Otherwise → fallback to CPU
     
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device : ", device)

Using device :  cpu


## Data Preproccesing

### Transforms

In [3]:
transform = transforms.Compose([
    # Converts PIL Image or NumPy array (pixel values 0–255) to a PyTorch tensor, Also scales pixel values from 0–255 → 0–1 (float)
    transforms.ToTensor(), 
    
    # Standardizes the tensor values to make training more stable and help the network converge faster, normalized_pixel=(pixel - mean)/std
    # This scales the pixel range from 0–1 → -1–1, which is often better for neural networks with activation functions like tanh or ReLU.
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

### Dataloaders

In [4]:
from torch.utils.data import DataLoader, Subset

# Download train data set
trainset = torchvision.datasets.MNIST(
    root="./data", # folder to store dataset
    train=True,  # training data
    download=True, # download if not exists
    transform=transform # apply preprocessing
)

# Download test data set
testset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create validations set from training set
train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size

train_data, val_data = random_split(trainset, [train_size, val_size])

# Only for training
trainloader = DataLoader(
    train_data,
    batch_size = 64,
    shuffle=True
)

# Only for validation / hp optimization
testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

# Only for evaluate final model
valloader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)

# Fast train loader
subset_indices = list(range(10000))  # use only first 10k samples
train_subset = Subset(train_data, subset_indices)

trainloader_fast = DataLoader(train_subset, batch_size=64, shuffle=True)

### Modified Le-Net CNN

In [5]:
class ModifiedLeNet(nn.Module):

    # constructor → define layers >>> batch(64 images at once) → conv → activation → pooling → fc → output
    def __init__(self):
        super().__init__()

        # ================= CONVOLUTIONAL BLOCK (FEATURE EXTRACTOR) ==================================

        # This part extracts image features
        self.conv_layers = nn.Sequential(
            # ------------- Conv layer 1 ---> INPUT > [batch, 1, 28, 28] -------------
            # input channels = 1 (grayscale) since we use MNIST, out_channels = number of filters
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3), # OUTPUT > [batch, 32, 26, 26] >>> batch = number of images processed at once, Batch size is defined in DataLoader
            # ReLU activation introduces non-linearity
            nn.ReLU(), # max(0, x)
            # MaxPool reduces image size by half
            nn.MaxPool2d(2), # [batch, 32, 13, 13]
    
            # ------------- Conv layer 2 ---> INPUT > [batch, 32, 13, 13] -------------
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3), # OUTPUT > [batch, 64, 11, 11]
            nn.ReLU(),
            nn.MaxPool2d(2), # [batch, 64, 5, 5]
    
            # ------------- Conv layer 3 (extra layer = modification) ---> INPUT > [batch, 64, 5, 5] -------------
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3), # OUTPUT > [batch, 128, 3, 3] # This is why you later use: 128 * 3 * 3
            nn.ReLU()
        )

        # ================= FULLY CONNECTED BLOCK (CLASSIFIER (Head))==================================

        # This part performs classification
        self.fc_layers = nn.Sequential(
            # ------------- Flatten layer ------------- 
            # converts 3D feature map → 1D vector, Flatten does NOT learn anything, It has no weights, It is just reshaping data
            nn.Flatten(), # [batch, 128, 3, 3] → [batch, 1152]
    
            # -------------Linear layer (dense layer) -------------
            # Compresses features into a compact representation, 128 feature maps * 3 * 3 size
            nn.Linear(128 * 3 * 3, 128),
    
            nn.ReLU(), # Adds non-linearity again.
    
            # ------------ Linear layer (Output layer) ------------
            # 10 neurons = digits 0–9
            nn.Linear(128, 10) # [batch, 10]
        )

    # forward pass defines data flow
    # input image > conv layers → feature maps > flatten > fully connected layers  > class scores
    '''
    This is the actual computation pipeline of the model, When we call > output = model(images), PyTorch internally does:model.forward(images)
    So this function is the execution path
    
    def forward(self, x): >>> x = input batch(shape : [batch, channels, height, width])
    x = self.conv_layers(x) >>> This sends the input through : Conv → ReLU → Pool → Conv → ReLU → Pool → Conv → ReLU, 
    So after this line > x = extracted feature maps
    x = self.fc_layers(x) >>> Now x goes into > Flatten → Linear → ReLU → Linear
    return x >>> Returns predictions > Example output for one image: [0.2, 1.1, 0.3, 5.4, 0.1, 0.0, 0.2, 0.1, 0.0, 0.2] 
    > Largest value index = predicted digit
    '''
    def forward(self, x):
        # pass input through conv layers
        x = self.conv_layers(x)
        # pass result through fully connected layers
        x = self.fc_layers(x)
        return x
         

### CREATE A FUNCTION TO TRAIN MODEL FOR Q-LEARNING ALGORITHM

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# epochs - number of rounds that model train on entire dataset
# lr - lr of cnn optimizer

def train_model(train_loader, device, model, lr, epochs, stop_acc=97):

    criterion = nn.CrossEntropyLoss() # loss function
    optimizer = optim.SGD(model.parameters(), lr=lr) # Optimizer

    loss_history = []
    acc_history = []

    # Train model for 1 epoch only for testing
    for epoch in range(epochs):

        # ---------------- TRAIN ----------------
        model.train()
        running_loss = 0

        # iterate over batches, When load MNIST using PyTorch dataset loaders, each batch gives: (images, labels) 64 sets(batches)
        # enumerate add index for each batch here we may have 938 batch all will be indexed as 1, 2, 3, ...
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() # summ all loss for each batch (938 losses), bcs we need entire dataset loss

            #if (i+1) % 100 == 0:  # print every 100 batches
            #    print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

            epoch_loss = running_loss / len(train_loader)
            loss_history.append(epoch_loss)

        # ---------------- EVALUATE ----------------
        model.eval()
        correct = 0
        total = 0
        
        # no gradients needed during testing
        with torch.no_grad():
            for images, labels in valloader:
                images, labels = images.to(device), labels.to(device)
                
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_acc = 100 * correct / total
        acc_history.append(epoch_acc)

        # print average loss per epoch
        print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}, Accuracy: {epoch_acc}")
        
        # ---------------- EARLY STOPPING ----------------        
        if epoch_acc >= stop_acc:
            print("*** Early stopping triggered — model already good!")
            break
            
        # print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")
        
    return model, loss_history, acc_history

### Define Q-Learning Table

In [7]:
import numpy as np
import random

# Define possible learning rates (discretized)
# these are the "states"

learning_rates = [0.0001, 0.001, 0.01, 0.1]

# action that can taken by agent
# -1: decrease, 0: keep, 1: increase

actions = [-1, 0, 1] 

batch_sizes = [32, 64, 128]

# Initialize Q-table: 
# rows = states, columns = actions
# len(learning_rates) → number of states
# Q-table keeps track of the expected reward for taking an action at a state
# at start Q-table is all zeros while someepisodes going it will fill(chekc later codes)

Q_table = np.zeros((len(learning_rates), len(actions)))

### Hyperparameters for Q-learning

In [8]:
alpha = 0.1   
# learning rate > how fast Q values update
# small = slow learning, stable
# big = fast learning, unstable

gamma = 0.9   
# discount factor > how much the agent cares about future rewards
# 0   = only care about immediate reward
# 1   = care about long-term reward
# 0.9 = good balance

epsilon = 1.0 
# exploration rate > probability of taking random action
# start with full exploration (100% random)

epsilon_decay = 0.95
# after each episode, exploration reduces
# agent becomes less random and more intelligent

epsilon_min = 0.05
# minimum exploration → agent still explores 5% forever
# prevents getting stuck in bad strategy

max_steps = 2
# maximum steps per episode (how long agent can move before episode ends)

### Helper Functions for Q-Learning

In [9]:
# Map learning rate to state index (return the index of passed lr value in the above defined list)
def get_state_index(lr):
    return learning_rates.index(lr)

# Return new action index & new learning rate based on action
# Ensures the learning rate stays within bounds with min(),max(), always increase/decrease by one step
# We passes index of a learning rate (row index) & action(-1, 0, 1, then retune new values
def take_action(state_index, action):
    if action == -1: # decrease
        new_index = max(0, state_index - 1) # 0, 1, 2
    elif action == 0: # keep same
        new_index = state_index # 0, 1, 2, 3
    else: # increase
        new_index = min(len(learning_rates)-1, state_index+1) # 3, 2, 1
    return new_index, learning_rates[new_index]

### Q-Learning Training Loop

In [16]:
# Use proper max_steps=1 – 3, epcohs=3 – 8, episodes=20 – 50 values

'''
---EPISODE---
One complete run of an agent from start -> until it stops
Episode = one full experience / one game / one attempt
Start -> agent acts -> gets rewards -> reaches goal or fails > END
That whole journey = 1 episode

if cnn epoches = 10, RL episodes = 20, steps = 25 >>> in each RL episode CNN train 25 times, like wise 20 times RL do same thing, in each CNN training
time it will train 10 times on entire dataset

---STEPS---
how long agent can move before episode ends
in each step CNN will be trained , retuern acc, update q table
'''

# ========== NOTE ==================
'''
until we find optimusm hps we use small training fat set , after found we trin again model with entire dat set and proper other valus, but now
epochs = 2-3 
steps = 2–4 
episodes = 20–50

entir dataset + epoch(10–20) + found lr
'''

# ========= Parameters ==============

episodes = 5  # how many trials the agent runs (20)
epochs= 3
max_steps= 3  # already define but for clarity

# Set table values to zero since i run several times this cell for testing
Q_table = np.zeros((len(learning_rates), len(actions)))

# Episodes × Steps = 5×10=50 times, The CNN training function is called 50 times, if cnn epochs=5, in each training data set used 5 times(5x10x5)

for episode in range(episodes):

    # Start with random lr
    lr = random.choice(learning_rates)
    state_index = get_state_index(lr)

    print(f"Episode {episode+1} started with random learning_rate = {lr} & state_index = {state_index}")
    
    # Train model and get reward(acc)
    _, losses, accuracies = train_model(
        train_loader=trainloader_fast,
        device=device, 
        model=model, 
        lr=lr,
        epochs=epochs
    )
    
    step = 1

    # Initialize a new CNN model for each episode
    model = ModifiedLeNet().to(device)
    
    # Do untill the reach max steps 1, 2, 3, ..., max_steps
    while step <= max_steps:
        
        print(f"EPOISODE {episode+1} STEP {step} started")

        
        # store previously trained model accuracy
        old_accuracy = accuracies[-1]
        
        # Choose action: epsilon-greedy
        if random.uniform(0, 1) < epsilon:
            # action index means the column number of the choosed action
            action_index = random.randint(0, len(actions)-1) # 0, 1, 2
            print(f"Explore action_index : {action_index} (0, 1, ,2)")
        else:
            # np.argmax([...]) returns the index of the maximum value in that Q_table[state_index] row
            # if all values are equal (all zeros), np.argmax returns index of the first occurrence of the maximum value(0)
            # action index means the column number of the choosed action
            action_index = np.argmax(Q_table[state_index]) # exploit, range 0-2, state index means row of the currently used lr 
            print(f"Exploit action_index : {action_index} (0, 1, ,2)")
            
        action = actions[action_index] # -1, 0, 1
        print(f"TAKEN ACTION : {action} (-1, 0, 1)")
        
        # let pass current index(this is actualy the state(lr) we pass its index instead value) & action choosen by above epsilon-greedy
        # then get new state index & learning rate
        new_state_index, new_lr = take_action(state_index, action) # state_index means current/ previous state's index
        print(f"new_state_index ={new_state_index} (old - {state_index})  , new_lr = {new_lr} (old - {lr})")

        # Train CNN with new learning rate
        _, losses, accuracies = train_model(
            trainloader_fast, 
            device, 
            model, 
            new_lr, 
            epochs
        )
        new_accuracy = accuracies[-1]
        
        # reward = new_accuracy - old_accuracy
        reward = new_accuracy / 100
        print(f"new_accuracy ={new_accuracy} %, reward = {reward}")

        # Update Q-table
        Q_value = Q_table[state_index, action_index] + alpha * (reward + gamma * np.max(Q_table[new_state_index])- Q_table[state_index, action_index])
        Q_table[state_index, action_index] = Q_value
        print(f"Q_value = {Q_value} added to ({state_index}, {action_index})")
        
        # move state forward
        state_index = new_state_index
        lr = new_lr
        accuracy = new_accuracy

        step += 1

        # print("Q-table:\n", Q_table)
        print(f"Step {episode+1}.{step-1}/{max_steps} done\n")
    
    # Prevent agent keeps exploring / exploit forever, decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    print("NEW EPSILON : ", epsilon)
        
    print(f"Episode {episode+1} DONE ( {max_steps} steps used), Final lr : {lr} & accuracy : {accuracy}, Reward: {reward:.2f}%")
    print("Q-table:\n", Q_table, "\n===============================\n")

print("================== DONE =====================")

Episode 1 started with random learning_rate = 0.0001 & state_index = 0
Epoch 1 Loss: 0.6873, Accuracy: 80.39166666666667
Epoch 2 Loss: 0.6832, Accuracy: 80.475
Epoch 3 Loss: 0.6784, Accuracy: 80.475
EPOISODE 1 STEP 1 started
Explore action_index : 1 (0, 1, ,2)
TAKEN ACTION : 0 (-1, 0, 1)
new_state_index =0 (old - 0)  , new_lr = 0.0001 (old - 0.0001)
Epoch 1 Loss: 2.3058, Accuracy: 12.4
Epoch 2 Loss: 2.3054, Accuracy: 12.566666666666666
Epoch 3 Loss: 2.3050, Accuracy: 12.841666666666667
new_accuracy =12.841666666666667 %, reward = -67.63333333333333
Q_value = -6.763333333333333 added to (0, 1)
Step 1.2/3 done

EPOISODE 1 STEP 2 started
Explore action_index : 1 (0, 1, ,2)
TAKEN ACTION : 0 (-1, 0, 1)
new_state_index =0 (old - 0)  , new_lr = 0.0001 (old - 0.0001)
Epoch 1 Loss: 2.3048, Accuracy: 13.116666666666667
Epoch 2 Loss: 2.3042, Accuracy: 13.391666666666667
Epoch 3 Loss: 2.3039, Accuracy: 13.908333333333333
new_accuracy =13.908333333333333 %, reward = 1.0666666666666664
Q_value = -5.

### Inspect Q-table

In [17]:
print("Q-table after training:")
for i, lr in enumerate(learning_rates):
    print(f"LR {lr}: {Q_table[i]}")

Q-table after training:
LR 0.0001: [ 0.1275     -5.98033333  2.8122218 ]
LR 0.001: [  0.01564167 -10.44209066  10.97839039]
LR 0.01: [-4.76163135  0.89878135  2.52917455]
LR 0.1: [-1.23891038  1.08694702  0.045     ]


### Select best learning rate

In [18]:
best_state = np.argmax(Q_table.max(axis=1))
best_lr = learning_rates[best_state]
print(f"Best learning rate found by Q-learning: {best_lr}")

Best learning rate found by Q-learning: 0.001


### Test model with new lr given by RL Agent

In [ ]:
learning_rate = best_lr
epochs = 20

model = ModifiedLeNet().to(device)

_, losses, accuracies = train_model(
    train_loader=trainloader,
    device=device,
    model=model, 
    lr=learning_rate, 
    epochs=epochs
)

print(f"Acc : {accuracies[-1]}")